In [61]:
import pandas as pd
from pathlib import Path

In [62]:
LOAD_PATH = Path("../../data/combined")

In [63]:
listing = pd.read_parquet(LOAD_PATH/"listing_fe.parquet")

In [64]:
def print_IQR(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    

    print(column,"---------")
    print("Q1: ", Q1)
    print("Q3: ", Q3)
    print("IQR: ", IQR)
    print("lower: ", lower)
    print("upper: ", upper)
    print(df[column].describe(percentiles=[.01,.05,.1,0.9,0.95,0.99,0.995,0.999]))
    print()

In [65]:
def percentile_columns(df, column, percentile):
    return df[column].quantile(percentile)

In [66]:
print_IQR(listing,"ClosePrice")

ClosePrice ---------
Q1:  620000.0
Q3:  1375000.0
IQR:  755000.0
lower:  -512500.0
upper:  2507500.0
count    1.371930e+05
mean     1.222312e+06
std      4.114386e+06
min      5.250000e+02
1%       2.500000e+05
5%       3.800000e+05
10%      4.500000e+05
90%      2.175000e+06
95%      2.950000e+06
99%      5.501148e+06
99.5%    7.211360e+06
99.9%    1.400000e+07
max      8.200000e+08
Name: ClosePrice, dtype: float64



In [7]:
listing.describe().T

,count,mean,min,25%,50%,75%,max,std
OriginalListPrice,519100.0,1421241.005955,0.0,599950.0,865000.0,1399888.0,1390000000.0,7237479.753117
CloseDate,157705,2025-05-03 09:14:17.740718,2024-01-01 00:00:00,2024-06-19 00:00:00,2025-08-29 00:00:00,2025-12-05 00:00:00,2030-04-16 00:00:00,NaN
ClosePrice,137193.0,1222312.217497,525.0,620000.0,875000.0,1375000.0,820000000.0,4114386.214326
Latitude,444987.0,34.594628,-117.472493,33.727424,34.043185,34.476709,737.0,1.994084
Longitude,444987.0,-118.483519,-124.22738,-118.726329,-118.004112,-117.237743,329.0,3.218157
LivingArea,519486.0,2019.710696,1.0,1287.0,1709.0,2344.0,17021321.0,23644.23637
ListPrice,519854.0,1338815.776765,100.0,599000.0,859000.0,1399000.0,195000000.0,2272833.09749
DaysOnMarket,519854.0,18.963769,0.0,5.0,11.0,22.0,583.0,25.886837
ParkingTotal,519834.0,8.062126,0.25,2.0,2.0,3.0,2593669.0,3597.512969
YearBuilt,519280.0,1981.136358,1776.0,1963.0,1981.0,2002.0,2028.0,26.244016


In [8]:
# More obvious outliers
outliers = [
    'GarageSpaces',
    'MainLevelBedrooms',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'ParkingTotal'
]
for out in outliers:
    print_IQR(listing, out)

GarageSpaces ---------
Q1:  2.0
Q3:  2.0
IQR:  0.0
lower:  2.0
upper:  2.0
count    496090.000000
mean          1.980364
std           3.986565
min           0.000000
1%            0.000000
5%            0.000000
10%           1.000000
90%           3.000000
95%           3.000000
99%           4.000000
99.5%         6.000000
99.9%        10.000000
max         600.000000
Name: GarageSpaces, dtype: float64

MainLevelBedrooms ---------
Q1:  1
Q3:  3
IQR:  2
lower:  -2.0
upper:  6.0
count    281457.0
mean     2.029759
std      6.559685
min           0.0
1%            0.0
5%            0.0
10%           0.0
90%           4.0
95%           4.0
99%           5.0
99.5%         6.0
99.9%         8.0
max        3333.0
Name: MainLevelBedrooms, dtype: Float64

BedroomsTotal ---------
Q1:  3
Q3:  4
IQR:  1
lower:  1.5
upper:  5.5
count    519824.0
mean     3.272592
std      1.132815
min           1.0
1%            1.0
5%            2.0
10%           2.0
90%           5.0
95%           5.0
99%     

count    519819.0
mean     2.677821
std      3.301024
min           1.0
1%            1.0
5%            1.0
10%           2.0
90%           4.0
95%           5.0
99%           7.0
99.5%         8.0
99.9%        11.0
max        2208.0
Name: BathroomsTotalInteger, dtype: Float64

ParkingTotal ---------
Q1:  2.0
Q3:  3.0
IQR:  1.0
lower:  0.5
upper:  4.5
count    5.198340e+05
mean     8.062126e+00
std      3.597513e+03
min      2.500000e-01
1%       1.000000e+00
5%       1.000000e+00
10%      1.000000e+00
90%      4.000000e+00
95%      6.000000e+00
99%      1.200000e+01
99.5%    1.500000e+01
99.9%    4.000000e+01
max      2.593669e+06
Name: ParkingTotal, dtype: float64



Looking at the IQR and percentages,

GarageSpaces, MainLevelBedrooms, BedroomsTotal, BathroomsTotalInteger, ParkingTotal numbers make a lot of sense at the 99th percentile. For these columns, everything after the 99th percentile will be removed

In [9]:
listing.describe().T

,count,mean,min,25%,50%,75%,max,std
OriginalListPrice,519100.0,1421241.005955,0.0,599950.0,865000.0,1399888.0,1390000000.0,7237479.753117
CloseDate,157705,2025-05-03 09:14:17.740718,2024-01-01 00:00:00,2024-06-19 00:00:00,2025-08-29 00:00:00,2025-12-05 00:00:00,2030-04-16 00:00:00,NaN
ClosePrice,137193.0,1222312.217497,525.0,620000.0,875000.0,1375000.0,820000000.0,4114386.214326
Latitude,444987.0,34.594628,-117.472493,33.727424,34.043185,34.476709,737.0,1.994084
Longitude,444987.0,-118.483519,-124.22738,-118.726329,-118.004112,-117.237743,329.0,3.218157
LivingArea,519486.0,2019.710696,1.0,1287.0,1709.0,2344.0,17021321.0,23644.23637
ListPrice,519854.0,1338815.776765,100.0,599000.0,859000.0,1399000.0,195000000.0,2272833.09749
DaysOnMarket,519854.0,18.963769,0.0,5.0,11.0,22.0,583.0,25.886837
ParkingTotal,519834.0,8.062126,0.25,2.0,2.0,3.0,2593669.0,3597.512969
YearBuilt,519280.0,1981.136358,1776.0,1963.0,1981.0,2002.0,2028.0,26.244016


In [53]:
outliers = [
    'LotSizeArea',
    'AssociationFee',
    'LivingArea'
]
for out in outliers:
    print_IQR(listing, out)

LotSizeArea ---------
Q1:  5001.0
Q3:  11761.0
IQR:  6760.0
lower:  -5139.0
upper:  21901.0
count    4.762510e+05
mean     3.768402e+04
std      1.170978e+06
min      0.000000e+00
1%       0.000000e+00
5%       5.000000e+00
10%      1.786000e+03
90%      4.335400e+04
95%      1.097025e+05
99%      4.360360e+05
99.5%    6.500625e+05
99.9%    1.905404e+06
max      3.084048e+08
Name: LotSizeArea, dtype: float64

AssociationFee ---------
Q1:  0.0
Q3:  400.0
IQR:  400.0
lower:  -600.0
upper:  1000.0
count    394261.000000
mean        269.645915
std        2220.823920
min           0.000000
1%            0.000000
5%            0.000000
10%           0.000000
90%         630.000000
95%         825.000000
99%        1800.000000
99.5%      2503.700000
99.9%      5955.000000
max      968348.000000
Name: AssociationFee, dtype: float64

LivingArea ---------
Q1:  1287.0
Q3:  2344.0
IQR:  1057.0
lower:  -298.5
upper:  3929.5
count    5.194860e+05
mean     2.019711e+03
std      2.364424e+04
min      

In [56]:
outliers = [
    'ClosePrice'
]
for out in outliers:
    print_IQR(listing, out)

ClosePrice ---------
Q1:  620000.0
Q3:  1375000.0
IQR:  755000.0
lower:  -512500.0
upper:  2507500.0
count    1.371930e+05
mean     1.222312e+06
std      4.114386e+06
min      5.250000e+02
1%       2.500000e+05
5%       3.800000e+05
10%      4.500000e+05
90%      2.175000e+06
95%      2.950000e+06
99%      5.501148e+06
99.5%    7.211360e+06
99.9%    1.400000e+07
max      8.200000e+08
Name: ClosePrice, dtype: float64



In [75]:
outlier_listing = listing.copy()
outlier_listing["Flagged"] = False

condition = outlier_listing["GarageSpaces"].gt(percentile_columns(outlier_listing,"GarageSpaces",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["MainLevelBedrooms"].gt(percentile_columns(outlier_listing,"MainLevelBedrooms",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["BedroomsTotal"].gt(percentile_columns(outlier_listing,"BedroomsTotal",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["BathroomsTotalInteger"].gt(percentile_columns(outlier_listing,"BathroomsTotalInteger",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["ParkingTotal"].gt(percentile_columns(outlier_listing,"ParkingTotal",.995)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition

condition = outlier_listing["LotSizeArea"].gt(percentile_columns(outlier_listing,"LotSizeArea",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["AssociationFee"].gt(percentile_columns(outlier_listing,"AssociationFee",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["LivingArea"].gt(percentile_columns(outlier_listing,"LivingArea",.999)).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition

condition = outlier_listing["ClosePrice"].le(0).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition
condition = outlier_listing["ClosePrice"].gt(2507500.0).fillna(False)
outlier_listing["Flagged"] = outlier_listing["Flagged"] | condition

print(outlier_listing[outlier_listing["Flagged"]].shape[0]/ outlier_listing.shape[0])

0.02664594289935251


In [76]:
listing_removed = outlier_listing.copy()

listing_removed = listing_removed.drop(listing_removed[listing_removed['Flagged']==True].index)

In [77]:
outlier_listing.shape

(519854, 44)

In [78]:
columns = [
    'GarageSpaces',
    'MainLevelBedrooms',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'ParkingTotal',
    'LotSizeArea',
    'AssociationFee',
    'LivingArea',
    'ClosePrice'
]

In [79]:
outlier_listing[columns].describe()

,GarageSpaces,MainLevelBedrooms,BedroomsTotal,BathroomsTotalInteger,ParkingTotal,LotSizeArea,AssociationFee,LivingArea,ClosePrice
count,496090.000000,281457.0,519824.0,519819.0,5.198340e+05,4.762510e+05,394261.000000,5.194860e+05,1.371930e+05
mean,1.980364,2.029759,3.272592,2.677821,8.062126e+00,3.768402e+04,269.645915,2.019711e+03,1.222312e+06
std,3.986565,6.559685,1.132815,3.301024,3.597513e+03,1.170978e+06,2220.823920,2.364424e+04,4.114386e+06
min,0.000000,0.0,1.0,1.0,2.500000e-01,0.000000e+00,0.000000,1.000000e+00,5.250000e+02
25%,2.000000,1.0,3.0,2.0,2.000000e+00,5.001000e+03,0.000000,1.287000e+03,6.200000e+05
50%,2.000000,2.0,3.0,2.0,2.000000e+00,7.200000e+03,135.000000,1.709000e+03,8.750000e+05
75%,2.000000,3.0,4.0,3.0,3.000000e+00,1.176100e+04,400.000000,2.344000e+03,1.375000e+06
max,600.000000,3333.0,72.0,2208.0,2.593669e+06,3.084048e+08,968348.000000,1.702132e+07,8.200000e+08


In [80]:
listing_removed.shape

(506002, 44)

In [81]:
listing_removed[columns].describe()

,GarageSpaces,MainLevelBedrooms,BedroomsTotal,BathroomsTotalInteger,ParkingTotal,LotSizeArea,AssociationFee,LivingArea,ClosePrice
count,483706.000000,276771.0,505972.0,505967.0,505983.000000,4.629850e+05,387108.000000,505699.000000,1.269150e+05
mean,1.916205,2.001843,3.240414,2.62327,2.633005,2.572266e+04,252.478518,1931.796681,9.631785e+05
std,0.839332,1.394502,1.07187,1.10984,1.676863,8.663794e+04,370.250861,1030.029245,4.935338e+05
min,0.000000,0.0,1.0,1.0,0.250000,0.000000e+00,0.000000,1.000000,5.250000e+02
25%,2.000000,1.0,3.0,2.0,2.000000,5.000000e+03,0.000000,1278.000000,6.000000e+05
50%,2.000000,2.0,3.0,2.0,2.000000,7.130000e+03,135.000000,1691.000000,8.339000e+05
75%,2.000000,3.0,4.0,3.0,3.000000,1.147900e+04,400.000000,2300.000000,1.240000e+06
max,10.000000,8.0,9.0,11.0,15.000000,1.905404e+06,5955.000000,12669.000000,2.507000e+06


In [82]:
listing_removed.to_parquet(LOAD_PATH/"listing_outliers_removed.parquet")